# Phase 1, Lesson 01 — Linear Algebra Intuition

> My from-scratch implementation. Read `docs/en.md` FIRST, then code without peeking at the course `code/`.

## Goal of this lesson (my words)
- Build `Vector` and `Matrix` classes from scratch (no NumPy) so the geometric meaning is visible.
- Re-derive dot product, projection, linear independence/rank, Gram–Schmidt by hand.
- Then compare against NumPy to confirm the from-scratch code is correct.

## My plan
1. `Vector` class: add/sub, dot, magnitude, normalize, cosine_similarity, angle_between
2. `Matrix` class: matmul (M@M and M@v), transpose, rank via row reduction
3. Free functions: `project`, `gram_schmidt`, `is_linearly_independent`
4. Exercises 1–6 from the lesson
5. Compare to NumPy / PyTorch

## 1. Build it from scratch

In [ ]:
import math


class Vector:
    def __init__(self, components):
        self.components = list(components)
        self.dim = len(self.components)

    def __add__(self, other):
        assert self.dim == other.dim
        return Vector([a + b for a, b in zip(self.components, other.components)])

    def __sub__(self, other):
        assert self.dim == other.dim
        return Vector([a - b for a, b in zip(self.components, other.components)])

    def __mul__(self, scalar):
        return Vector([x * scalar for x in self.components])

    __rmul__ = __mul__

    def dot(self, other):
        assert self.dim == other.dim
        return sum(a * b for a, b in zip(self.components, other.components))

    def magnitude(self):
        return math.sqrt(sum(x ** 2 for x in self.components))

    def normalize(self):
        mag = self.magnitude()
        assert mag > 1e-12, "cannot normalize the zero vector"
        return Vector([x / mag for x in self.components])

    def cosine_similarity(self, other):
        return self.dot(other) / (self.magnitude() * other.magnitude())

    def angle_between(self, other):
        # Exercise 1: angle in degrees
        cos = self.cosine_similarity(other)
        cos = max(-1.0, min(1.0, cos))  # guard against fp overshoot
        return math.degrees(math.acos(cos))

    def __repr__(self):
        return f"Vector({self.components})"


# quick smoke test
a = Vector([1, 2, 3])
b = Vector([4, 5, 6])
print(f"a + b   = {a + b}")
print(f"a · b   = {a.dot(b)}")
print(f"|a|     = {a.magnitude():.4f}")
print(f"cos(a,b)= {a.cosine_similarity(b):.4f}")
print(f"angle   = {a.angle_between(b):.2f}°")

In [ ]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows]
        self.shape = (len(self.rows), len(self.rows[0]))

    def __matmul__(self, other):
        # M @ v  and  M @ M
        if isinstance(other, Vector):
            assert self.shape[1] == other.dim
            return Vector([
                sum(self.rows[i][j] * other.components[j] for j in range(self.shape[1]))
                for i in range(self.shape[0])
            ])
        assert self.shape[1] == other.shape[0]
        out = []
        for i in range(self.shape[0]):
            row = []
            for j in range(other.shape[1]):
                row.append(sum(
                    self.rows[i][k] * other.rows[k][j]
                    for k in range(self.shape[1])
                ))
            out.append(row)
        return Matrix(out)

    def transpose(self):
        return Matrix([
            [self.rows[j][i] for j in range(self.shape[0])]
            for i in range(self.shape[1])
        ])

    def rank(self):
        # Gaussian elimination on a copy → count pivots
        rows = [row[:] for row in self.rows]
        n_rows, n_cols = len(rows), len(rows[0])
        r = 0
        col = 0
        while r < n_rows and col < n_cols:
            pivot = None
            for row in range(r, n_rows):
                if abs(rows[row][col]) > 1e-10:
                    pivot = row
                    break
            if pivot is None:
                col += 1
                continue
            rows[r], rows[pivot] = rows[pivot], rows[r]
            scale = rows[r][col]
            rows[r] = [x / scale for x in rows[r]]
            for row in range(n_rows):
                if row != r and abs(rows[row][col]) > 1e-10:
                    factor = rows[row][col]
                    rows[row] = [rows[row][j] - factor * rows[r][j] for j in range(n_cols)]
            r += 1
            col += 1
        return r

    def __repr__(self):
        return f"Matrix({self.rows})"


# rotation 90° counter-clockwise
R = Matrix([[0, -1], [1, 0]])
p = Vector([3, 1])
print(f"rotate [3,1] by 90° → {R @ p}")

# a neural-network-like layer: 3D → 2D
W = Matrix([[0.1, -0.2, 0.3], [0.4, 0.5, -0.1]])
x = Vector([1.0, 0.5, -0.3])
print(f"W @ x (a fake NN layer) → {W @ x}")

In [ ]:
def project(a, b):
    # proj_b(a) = (a·b / b·b) * b
    scalar = a.dot(b) / b.dot(b)
    return scalar * b


def is_linearly_independent(vectors):
    # build matrix whose rows are the vectors, rank must equal count of vectors
    m = Matrix([v.components[:] for v in vectors])
    return m.rank() == len(vectors)


def gram_schmidt(vectors):
    # produce an orthonormal basis
    orthonormal = []
    for v in vectors:
        w = v
        for u in orthonormal:
            w = w - project(w, u)
        if w.magnitude() < 1e-10:
            # v was linearly dependent on the previous ones
            continue
        orthonormal.append(w.normalize())
    return orthonormal


# smoke test Gram–Schmidt
v1, v2, v3 = Vector([1, 0, 0]), Vector([1, 1, 0]), Vector([1, 1, 1])
basis = gram_schmidt([v1, v2, v3])
for i, u in enumerate(basis, 1):
    print(f"u{i} = {u}   |u{i}| = {u.magnitude():.6f}")
print(f"u1·u2 = {basis[0].dot(basis[1]):.2e}  (≈0 → orthogonal ✓)")
print(f"u1·u3 = {basis[0].dot(basis[2]):.2e}")
print(f"u2·u3 = {basis[1].dot(basis[2]):.2e}")

## 2. Exercises (1–6 from the lesson)

In [ ]:
# Ex 1 — angle between two vectors
print("Ex 1:", Vector([1, 0]).angle_between(Vector([0, 1]), ), "° (expect 90)")
print("Ex 1:", Vector([1, 0]).angle_between(Vector([1, 1]), ), "° (expect 45)")

In [ ]:
# Ex 2 — scaling matrix 2x, 3y applied to [1,1]
S = Matrix([[2, 0], [0, 3]])
print("Ex 2: scale [1,1] by diag(2,3) →", S @ Vector([1, 1]), "(expect [2,3])")

In [ ]:
# Ex 3 — 5 random 50-D vectors, find the two most similar by cosine
import random
random.seed(0)
vecs = [Vector([random.gauss(0, 1) for _ in range(50)]) for _ in range(5)]
best = None
for i in range(5):
    for j in range(i + 1, 5):
        sim = vecs[i].cosine_similarity(vecs[j])
        if best is None or sim > best[0]:
            best = (sim, i, j)
print(f"Ex 3: most similar pair = vec#{best[1]} & vec#{best[2]}, cosine = {best[0]:.4f}")

In [ ]:
# Ex 4 — verify Gram–Schmidt output is orthonormal
basis = gram_schmidt([Vector([2, 0, 0]), Vector([1, 2, 0]), Vector([1, 1, 1])])
unit_ok = all(abs(u.magnitude() - 1) < 1e-9 for u in basis)
orth_ok = all(
    abs(basis[i].dot(basis[j])) < 1e-9
    for i in range(len(basis)) for j in range(i + 1, len(basis))
)
print(f"Ex 4: all unit length? {unit_ok}   all pairwise orthogonal? {orth_ok}")

In [ ]:
# Ex 5 — a 3x3 matrix of rank 2; what do its columns span?
# column 3 = column 1 + 2*column 2  →  dependent
A = Matrix([
    [1, 0, 1],
    [0, 1, 2],
    [0, 0, 0],
])
print(f"Ex 5: rank = {A.rank()} (expect 2)")
print("       columns span the xy-plane (a 2D subspace of R³).")

In [ ]:
# Ex 6 — project [1,2,3] onto [1,1,1]; what does the result mean?
a = Vector([1, 2, 3])
d = Vector([1, 1, 1])
p = project(a, d)
resid = a - p
print(f"Ex 6: proj = {p}")
print(f"       residual = {resid}, a·residual = {a.dot(resid):.2e}, d·resid = {d.dot(resid):.2e} (≈0 → residual ⊥ d ✓)")
print("       meaning: the part of [1,2,3] that points along the diagonal [1,1,1];")
print("       the residual is the 'unique' component of a, perpendicular to the diagonal.")

## 3. Compare against NumPy / PyTorch

In [ ]:
import numpy as np

a_np = np.array([1, 2, 3], dtype=float)
b_np = np.array([4, 5, 6], dtype=float)

print("dot:   mine=", Vector([1, 2, 3]).dot(Vector([4, 5, 6])),
      " numpy=", float(np.dot(a_np, b_np)))
print("norm:  mine=", f"{Vector([1, 2, 3]).magnitude():.6f}",
      " numpy=", f"{np.linalg.norm(a_np):.6f}")

# projection of [1,2,3] onto [1,1,1]
a, d = Vector([1, 2, 3]), Vector([1, 1, 1])
print("proj:  mine=", project(a, d).components,
      " numpy=", ((np.dot([1, 2, 3], [1, 1, 1]) / np.dot([1, 1, 1], [1, 1, 1])) * np.array([1, 1, 1])).tolist())

# QR decomposition vs my Gram–Schmidt
V = np.array([[2., 1, 1], [0, 2, 1], [0, 0, 1]])  # columns = the input vectors
Q, R = np.linalg.qr(V)
print("numpy Q (first basis vector) ≈", Q[:, 0].tolist())
print("mine  u1                  =", gram_schmidt([Vector([2, 0, 0]), Vector([1, 2, 0]), Vector([1, 1, 1])])[0].components)

In [ ]:
import torch

x = torch.randn(3, requires_grad=True)
y = torch.tensor([1.0, 0.0, 0.0])
sim = torch.dot(x, y)
sim.backward()
print(f"d(dot(x,y))/dx = {x.grad}  ← equals y; gradient of a·y wrt x is just y.")

## 4. What I learned (30-sec explanation to myself)

- **Vector** = list of numbers = a point/direction. Magnitude is Pythagoras; adding is component-wise; scaling stretches or flips.
- **Matrix** = a transformation. `M @ v` is literally one neural-network layer. Weights *are* matrices.
- **Dot product** = similarity. `>0` aligned, `=0` unrelated, `<0` opposite. Cosine similarity normalizes by length → this is how RAG/attention measure relevance.
- **Linear independence / rank** = how much real information is in the columns. Redundant features → rank-deficient → unstable weights. LoRA exploits low rank to fine-tune cheaply.
- **Projection** = the shadow of one vector along another; residual is orthogonal. Regression/PCA/attention are all projections.
- **Gram–Schmidt** = turn any independent set into an orthonormal basis (basis for QR, stable numerics).

Next: lesson 02 (week-1 plan).